# TP 3 : décortiquer puis exécuter un job MapReduce

**Module Big Data (Hadoop & Spark), Bac+5, Institut GoldenCollar. Mouaad AGOURRAM.** Durée : 55 minutes.

## Contexte

Le cluster du module tourne dans Docker sur votre machine : un maître (`master`), trois travailleurs
(`worker1` à `worker3`) et un poste client (`client`). Ce notebook tourne hors du cluster, dans votre
environnement Python, et pilote Hadoop par le conteneur client.

Vous travaillez sur les logs HTTP du serveur web du Kennedy Space Center (NASA) d'août 1995, déjà vus : une
ligne par requête reçue, au format suivant.

```
hôte - - [jour/mois/année:heure fuseau] "requête" code octets
```

L'avant-dernier champ est le code de réponse HTTP. Le dernier est le nombre d'octets envoyés ; il vaut « - »
quand aucun octet n'a été envoyé. La page de description
(https://ita.ee.lbl.gov/html/contrib/NASA-HTTP.html) demande de ne pas analyser ces logs au-delà des
tendances générales de trafic : vous calculez des volumes par jour, par code de réponse et par ressource,
jamais par hôte.

## Objectifs

1. Dérouler à la main, en Python, les étapes map, shuffle et reduce d'un job.
2. Écrire un mapper et un reducer pour Hadoop Streaming, les tester par un pipe, puis les lancer sur le
   cluster.
3. Lire les compteurs d'un job, et vérifier son résultat contre un calcul sur une seule machine.
4. Mesurer l'effet d'un combiner, et écrire une moyenne distribuée juste.
5. Mesurer le coût fixe d'un job.

## Consignes

- Environnement : depuis la racine du module, `uv sync --group tp`, puis choisissez `.venv` comme noyau de
  ce notebook. Ouvrez-le depuis le dossier `seance3/tp`. Le cluster doit tourner (`docker compose up -d`),
  avec ses trois DataNodes et le serveur d'historique MapReduce sur http://localhost:19888. Si cette page
  ne répond pas, reconstruisez l'image depuis la racine du module (`docker compose up -d --build`), puis
  attendez une minute.
- La fonction `cluster(commande)`, définie dans la préparation, exécute une commande shell dans le conteneur
  client, où sont installés `hdfs` et `mapred`, et renvoie la liste des lignes de sa sortie. La fonction
  `job(commande)` lance une commande de job (`mapred streaming ...`) dans le conteneur client, et renvoie
  son journal complet, tel que la commande l'affiche. La fonction `compteurs(journal)` extrait de ce journal
  les compteurs affichés à la fin du job : `{nom: valeur}`.
- Le dossier du module est monté dans le conteneur client sous `/cours` : un fichier écrit dans
  `seance3/tp` sur votre machine y est `/cours/seance3/tp`. Écrivez vos mappers et reducers dans ce
  dossier, avec une cellule `%%writefile nom.py`.
- Vos scripts tournent sous `python3` (Python 3.10) dans les conteneurs, réglés en UTF-8. Une dizaine de
  lignes d'août contiennent des octets qui ne sont pas de l'UTF-8 : lisez l'entrée standard en latin-1
  (`sys.stdin.reconfigure(encoding="latin-1")`), et le fichier local aussi (`encoding="latin-1"`).
- Pour un test par un pipe, triez avec `LC_ALL=C sort` : c'est l'ordre octet par octet de Hadoop.
- Dans une commande `mapred streaming`, les options génériques (`-D ...`) se placent avant les autres.
- Chaque question se termine par un livrable : affectez la valeur demandée à la variable indiquée.
- La question 9 est un bonus.

## Préparation

La cellule suivante définit `cluster`, `job` et `compteurs`, télécharge les logs d'août 1995 dans `data/`
(16 Mo compressés) et les décompresse. Elle écrit aussi dans `data/extrait.log` les lignes 499 722 à
499 729 du fichier, qui servent à la partie A. Exécutez-la sans la modifier.

In [ ]:
import gzip
import re
import shutil
import subprocess
import time
import urllib.request
from collections import Counter
from itertools import islice
from pathlib import Path

COMPOSE = Path.cwd().parents[1] / "docker-compose.yml"
assert COMPOSE.exists(), f"{COMPOSE} introuvable : ouvrez ce notebook depuis le dossier seance3/tp"
CLIENT = "/cours/seance3/tp"          # le dossier seance3/tp, vu depuis le conteneur client


def cluster(commande):
    # Exécute une commande shell dans le conteneur client du cluster ; renvoie ses lignes de sortie
    resultat = subprocess.run(
        ["docker", "compose", "-f", str(COMPOSE), "exec", "-T", "client", "bash", "-c", commande],
        capture_output=True, text=True)
    if resultat.returncode != 0:
        raise RuntimeError(f"Échec de : {commande}\n{resultat.stderr}")
    return resultat.stdout.splitlines()


def job(commande):
    # Lance un job dans le conteneur client ; renvoie son journal complet (sortie et erreurs)
    resultat = subprocess.run(
        ["docker", "compose", "-f", str(COMPOSE), "exec", "-T", "client", "bash", "-c", f"( {commande} ) 2>&1"],
        capture_output=True, text=True)
    if resultat.returncode != 0:
        fin = "\n".join(resultat.stdout.splitlines()[-40:])
        raise RuntimeError(f"Échec du job : {commande}\n{fin}")
    return resultat.stdout


def compteurs(journal):
    # Les compteurs affichés à la fin du journal d'un job : {nom: valeur}
    return {m.group(1).strip(): int(m.group(2))
            for m in re.finditer(r"^\s+([^=\n]+?)\s*=(\d+)\s*$", journal, re.M)}


DATA = Path("data")
DATA.mkdir(exist_ok=True)
archive = DATA / "NASA_access_log_Aug95.gz"
if not archive.exists():
    urllib.request.urlretrieve("https://ita.ee.lbl.gov/traces/NASA_access_log_Aug95.gz", archive)
AOUT = DATA / "NASA_access_log_Aug95"
if not AOUT.exists():
    with gzip.open(archive, "rb") as entree, open(AOUT, "wb") as sortie:
        shutil.copyfileobj(entree, sortie)
print(AOUT, f"{AOUT.stat().st_size:,} octets".replace(",", " "))

with open(AOUT, encoding="latin-1") as f:
    EXTRAIT = list(islice(f, 499721, 499729))      # lignes 499 722 à 499 729
(DATA / "extrait.log").write_text("".join(EXTRAIT), encoding="latin-1")
print("".join(EXTRAIT), end="")

## Partie A : à la main, en Python

### Question 1 : la fonction map

On veut calculer, pour chaque jour du mois, le nombre de requêtes et les octets envoyés. Écrivez la fonction
`map_ligne(ligne)` qui renvoie la liste des couples (clé, valeur) que map émet pour une ligne de log : la clé
est le jour du mois, sur deux chiffres (`"11"`, `"12"`...), la valeur est le nombre d'octets envoyés, un
entier. Une ligne illisible ne produit aucun couple. Appliquez-la aux huit lignes de `EXTRAIT`.

**Livrable** : la fonction `map_ligne`, et `q1_couples` (liste des couples émis pour les huit lignes, dans
l'ordre des lignes).

In [ ]:
# Question 1

### Question 2 : le shuffle et la fonction reduce

Écrivez la fonction `shuffle(couples)`, qui regroupe les valeurs par clé comme le fait le shuffle : elle
renvoie un dictionnaire `{clé: [valeurs]}`, dont les clés sont triées. Écrivez ensuite la fonction
`reduce_jour(cle, valeurs)`, qui renvoie le triplet `(jour, nombre de requêtes, octets envoyés)`.
Appliquez-les aux couples de la question 1.

**Livrable** : `q2_groupes` (le résultat de `shuffle` sur `q1_couples`) et `q2_resultat` (la liste des
triplets renvoyés par `reduce_jour`, un par clé, dans l'ordre des clés).

In [ ]:
# Question 2

## Partie B : Hadoop Streaming sur le cluster

### Question 3 : le mapper et le reducer

Écrivez `mapper_jour.py` et `reducer_jour.py` dans `seance3/tp`, pour le même calcul qu'en partie A, au
contrat de Hadoop Streaming : le mapper lit les lignes de log sur l'entrée standard et écrit des lignes
« clé TAB valeur » ; le reducer lit ces lignes, triées par clé, et écrit une ligne
« jour TAB requêtes TAB octets » par jour. Testez-les par un pipe, dans le conteneur client, sur les huit
lignes de `data/extrait.log`.

**Livrable** : `q3_sortie` (liste des lignes écrites par le reducer) et `q3_egal_q2` (booléen : même
résultat qu'à la question 2).

In [ ]:
%%writefile mapper_jour.py
# Question 3 : le mapper

In [ ]:
%%writefile reducer_jour.py
# Question 3 : le reducer

In [ ]:
# Question 3 : le test par un pipe

### Question 4 : le job sur le cluster

Déposez les logs d'août dans HDFS, dans `/tp3/aout/aout95.log`, avec des blocs de 64 Mo (67 108 864 octets).
Lancez votre job avec Hadoop Streaming et 2 reducers, avec la sortie dans `/tp3/sorties/jour`, puis lisez le
résultat dans HDFS. Comparez-le au calcul de la partie A (`map_ligne`, `shuffle`, `reduce_jour`) appliqué à
tout le fichier local.

**Livrable** : `q4_maps` (nombre de tâches map du job, lu dans son journal), `q4_fichiers` (nombre de
fichiers de résultat), `q4_jours` (nombre de jours dans le résultat) et `q4_egal_python` (booléen : le
résultat du cluster est-il exactement celui du calcul en Python ?).

In [ ]:
# Question 4

### Question 5 : les compteurs

Dans les compteurs du job de la question 4, relevez `Launched map tasks`, `Map input records`,
`Map output records`, `Shuffled Maps` et `Reduce input groups`. Combien de lignes compte le fichier local ?
Toutes les lignes lues ont-elles produit un couple ? Le nombre de transferts du shuffle est-il celui que
vous attendiez ?

**Livrable** : `q5_compteurs` (dictionnaire des cinq compteurs), `q5_lignes` (entier : lignes du fichier
local), `q5_tout_emis` (booléen : chaque ligne a-t-elle produit un couple ?) et `q5_shuffle_attendu`
(entier : le nombre de transferts que vous calculez à partir du nombre de maps et de reducers).

In [ ]:
# Question 5

## Partie C : le combiner et la moyenne

### Question 6 : le combiner

Relancez le job de la question 4 en utilisant votre reducer comme combiner, avec la sortie dans
`/tp3/sorties/jour_combiner`. Relevez les octets envoyés par le shuffle (compteur `Reduce shuffle bytes`),
avec et sans combiner. Le résultat est-il identique ?

**Livrable** : `q6_shuffle_sans` et `q6_shuffle_avec` (entiers, en octets), `q6_divise_par` (le rapport
des deux, arrondi à une décimale) et `q6_meme_resultat` (booléen).

In [ ]:
# Question 6

### Question 7 : une moyenne juste

Calculez, pour chaque code de réponse HTTP, la taille moyenne d'une réponse en octets (« - » compte pour 0),
avec un job Hadoop Streaming à 2 reducers **et un combiner**. Écrivez les scripts nécessaires dans
`seance3/tp`, avec la sortie dans `/tp3/sorties/moyenne`. Vérifiez dans les compteurs que le combiner a
bien reçu des enregistrements, puis comparez au même calcul en Python sur le fichier local.

**Livrable** : `q7_moyennes` (dictionnaire `{code: moyenne}`, moyennes arrondies à 2 décimales),
`q7_combine_input` (entier : compteur `Combine input records`) et `q7_egal_python` (booléen).

In [ ]:
# Question 7

## Partie D : le coût fixe

### Question 8 : dix lignes ou un mois

Déposez les 10 premières lignes d'août dans `/tp3/petit/dix_lignes.log`. Lancez dessus le job de la
question 4, avec la sortie dans `/tp3/sorties/petit`, puis relancez le job de la question 4 sur le mois
entier. Mesurez depuis le notebook la durée de chacun des deux jobs, de bout en bout, avec
`time.perf_counter()`. Quelle part du temps du mois coûtent dix lignes ?

**Livrable** : `q8_duree_10` et `q8_duree_mois` (secondes, arrondies à une décimale) et `q8_part`
(entier : la durée des dix lignes en pourcentage de celle du mois).

In [ ]:
# Question 8

## Bonus

### Question 9 (bonus) : les 10 ressources les plus demandées

La ressource demandée est le deuxième mot de la requête, entre guillemets (`/ksc.html` dans
`"GET /ksc.html HTTP/1.0"`) ; une requête sans deuxième mot est ignorée. Trouvez les 10 ressources les plus
demandées du mois avec deux jobs Hadoop Streaming enchaînés : le premier compte les requêtes par ressource,
le second lit la sortie du premier dans HDFS et ne garde que les 10 plus demandées. En cas d'égalité, l'ordre
alphabétique de la ressource départage. Comparez au même calcul en Python sur le fichier local.

**Livrable** : `q9_top10` (liste de 10 couples `(ressource, requêtes)`, du plus demandé au moins demandé) et
`q9_egal_python` (booléen).

In [ ]:
# Question 9

## Fin du TP

Gardez le cluster et vos scripts : `docker compose stop` arrête le cluster sans rien effacer. Si vous avez
fini avant la fin du créneau, ouvrez l'historique MapReduce (http://localhost:19888), choisissez votre job
de la question 6, et repérez dans Counters les compteurs `Combine input records` et `Reduce shuffle bytes`.